# Semana 01 · AgentOps y LLMOps con MLflow Tracing

**Solución docente.** Aplicar trazas y evaluación al asistente didáctico que acompaña el caso de predicción cardiovascular del repositorio. Por defecto no llama a ningún LLM, por lo que funciona en Databricks Free Edition sin coste de inferencia.

> No envíes datos de pacientes, secretos ni información personal a las trazas o a un endpoint. Este agente no ofrece consejo clínico.

## 0. Dependencias

Ejecuta la instalación sólo si el runtime no proporciona MLflow 3.1 o superior. Si se actualiza el entorno, reinicia Python y continúa desde la configuración. Para activar la extensión opcional con endpoint instala también `databricks-openai` con `%pip install -q --upgrade databricks-openai`.

In [ ]:
%pip install -q --upgrade "mlflow[databricks]>=3.1"

## 1. Configuración segura

Free Edition usa compute serverless con cuota. El modo determinista es obligatorio y reproducible. La llamada a Foundation Model es una demostración docente breve: permite observar un span de LLM y el uso de tokens sin convertirla en condición de la práctica.

In [ ]:
from typing import Any

import mlflow
from mlflow.genai.scorers import scorer

USE_LLM = False
MODEL_ENDPOINT = "REEMPLAZA_CON_UN_ENDPOINT_AUTORIZADO"
mlflow.set_tracking_uri("databricks")

## 2. Agente mínimo y trazas explícitas

El agente primero clasifica la pregunta y después recupera contexto autorizado. Los decoradores `@mlflow.trace` registran esos pasos anidados. En un agente real también aparecerían llamadas al LLM, recuperadores y herramientas externas.

In [ ]:
COURSE_CONTEXT = {
    "mlflow": (
        "Un experimento de MLflow agrupa runs relacionados y permite comparar "
        "parámetros, métricas, artefactos, trazas y evaluaciones."
    ),
    "security": (
        "Nunca pegues tokens, correos, datos sensibles ni prompts con información "
        "privada en un notebook, un parámetro, un artefacto o una traza."
    ),
    "risk": (
        "El caso cardiovascular es didáctico: no ofrece consejo clínico. Antes de "
        "producción documenta riesgos, propietarios, mitigaciones y evaluación."
    ),
}


@mlflow.trace
def route_question(question: str) -> str:
    normalized = question.lower()
    if any(word in normalized for word in ("token", "secreto", "privacidad")):
        return "security"
    if any(word in normalized for word in ("riesgo", "producción", "desplegar", "clínic")):
        return "risk"
    return "mlflow"


@mlflow.trace
def retrieve_course_context(topic: str) -> str:
    return COURSE_CONTEXT[topic]


@mlflow.trace
def course_agent(question: str) -> str:
    topic = route_question(question)
    context = retrieve_course_context(topic)

    if not USE_LLM:
        return f"Respuesta trazable (modo determinista): {context}"

    if MODEL_ENDPOINT.startswith("REEMPLAZA_"):
        raise ValueError("Configura un endpoint autorizado antes de activar USE_LLM.")

    from databricks_openai import DatabricksOpenAI

    mlflow.openai.autolog()
    client = DatabricksOpenAI()
    response = client.chat.completions.create(
        model=MODEL_ENDPOINT,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Eres un asistente didáctico de Operación de Modelos. "
                    "Responde sólo con el contexto proporcionado. Si falta "
                    "información, di que no tienes confirmación."
                ),
            },
            {"role": "system", "content": f"Contexto autorizado: {context}"},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content


for question in [
    "¿Qué objeto agrupa varios runs?",
    "¿Puedo pegar mi token en un parámetro de MLflow?",
    "¿Este caso permite tomar una decisión clínica?",
]:
    print(f"P: {question}\nR: {course_agent(question)}\n")

## 3. Demo docente: añadir un span de LLM real

1. En la barra lateral de Free Edition abre **Serving** y localiza los Foundation Model APIs que aparecen en la parte superior. Copia el nombre de un endpoint de chat disponible; no lo inventes ni crees uno nuevo.
2. Ejecuta `%pip install -q --upgrade databricks-openai`, reinicia Python y vuelve a ejecutar las secciones 1 y 2.
3. Asigna el nombre copiado a `MODEL_ENDPOINT`, cambia `USE_LLM = True` y ejecuta **una** pregunta breve. `DatabricksOpenAI` reutiliza la identidad del notebook; no pegues un token.
4. En **Experiments**, abre la nueva traza: además de `route_question` y `retrieve_course_context` aparecerá el span de OpenAI/endpoint, con latencia y uso de tokens si el endpoint lo devuelve.

Si el endpoint no está disponible o se ha alcanzado la cuota, vuelve a `USE_LLM = False`; el resto de la práctica sigue siendo válido.

## 4. Inspección de la traza

Abre **Experiments** en la barra lateral y selecciona una traza de `course_agent`. Identifica la pregunta, la ruta elegida, el contexto recuperado y la respuesta. Si una respuesta fuese incorrecta, esos pasos permiten localizar el origen.

## 5. LLMOps: evaluación reproducible

El *scorer* siguiente comprueba una señal mínima y no invoca un juez LLM. Sirve para enseñar el ciclo datos de evaluación → traza → evaluación → evidencia. No demuestra por sí solo calidad, seguridad ni utilidad.

In [ ]:
EVALUATION_DATA = [
    {
        "inputs": {"question": "¿Qué objeto agrupa varios runs?"},
        "expectations": {"must_include": "experimento"},
    },
    {
        "inputs": {"question": "¿Puedo registrar tokens en MLflow?"},
        "expectations": {"must_include": "tokens"},
    },
    {
        "inputs": {"question": "¿Este caso permite tomar una decisión clínica?"},
        "expectations": {"must_include": "no ofrece consejo clínico"},
    },
]


@scorer
def contains_expected_phrase(
    *,
    outputs: str | None,
    expectations: dict[str, Any] | None,
) -> bool:
    if not outputs or not expectations:
        return False
    return expectations["must_include"].lower() in outputs.lower()


evaluation = mlflow.genai.evaluate(
    data=EVALUATION_DATA,
    predict_fn=course_agent,
    scorers=[contains_expected_phrase],
)
print(f"Run de evaluación: {evaluation.run_id}")

## Debrief

1. ¿Qué riesgo detecta el scorer y cuáles deja fuera?
2. ¿Qué contenido no debería persistir en una traza de producción?
3. Si habilitas `USE_LLM`, ¿qué llamada, coste y latencia se añaden a la traza?
4. Si no hay cuota para un endpoint, ¿qué partes del flujo continúan siendo reproducibles?